# 03. SBERT Baseline

Sentence-BERT embeddings + handcrafted features.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sentence_transformers import SentenceTransformer
import torch
from pathlib import Path
import gc

DATA_DIR = Path('../data')
OUTPUT_DIR = Path('../output')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

train_df['target'] = (train_df['winner_model_a'].astype(int) * 0 +
                      train_df['winner_model_b'].astype(int) * 1 +
                      train_df['winner_tie'].astype(int) * 2)

In [ ]:
model_name = 'all-MiniLM-L6-v2'
print(f'Loading {model_name}...')
sbert = SentenceTransformer(model_name, device=DEVICE)

In [ ]:
def encode_pairs(texts_a, texts_b, prompts, sbert_model, batch_size=64):
    """Encode response pairs with SBERT."""
    emb_a = sbert_model.encode(texts_a.fillna(''), show_progress_bar=True, batch_size=batch_size)
    emb_b = sbert_model.encode(texts_b.fillna(''), show_progress_bar=True, batch_size=batch_size)

    # Cosine similarity
    norm_a = emb_a / np.linalg.norm(emb_a, axis=1, keepdims=True)
    norm_b = emb_b / np.linalg.norm(emb_b, axis=1, keepdims=True)
    cos_sim = np.sum(norm_a * norm_b, axis=1)

    # Difference
    diff = emb_a - emb_b

    # Handcrafted features
    len_a = np.array([len(str(s)) for s in texts_a])
    len_b = np.array([len(str(s)) for s in texts_b])
    len_ratio = len_a / (len_b + 1)
    word_count_a = np.array([len(str(s).split()) for s in texts_a])
    word_count_b = np.array([len(str(s).split()) for s in texts_b])
    word_ratio = word_count_a / (word_count_b + 1)
    hand = np.column_stack([len_a, len_b, len_ratio, word_count_a, word_count_b, word_ratio])

    features = np.concatenate([emb_a, emb_b, diff, cos_sim[:, None], hand], axis=1)
    return features

print('Encoding train...')
X_train = encode_pairs(train_df['response_a'], train_df['response_b'], train_df['prompt'], sbert)
print(f'Train features shape: {X_train.shape}')

print('Encoding test...')
X_test = encode_pairs(test_df['response_a'], test_df['response_b'], test_df['prompt'], sbert)
print(f'Test features shape: {X_test.shape}')

In [ ]:
y_train = train_df['target'].values

# CV evaluation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros((len(X_train), 3))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    model = LogisticRegression(max_iter=2000, C=1.0, solver='lbfgs', multi_class='multinomial')
    model.fit(X_train[train_idx], y_train[train_idx])
    oof_preds[val_idx] = model.predict_proba(X_train[val_idx])
    fold_loss = log_loss(y_train[val_idx], oof_preds[val_idx])
    print(f'Fold {fold+1}: log_loss={fold_loss:.4f}')

oof_loss = log_loss(y_train, oof_preds)
print(f'\nOOF log_loss: {oof_loss:.4f}')

In [ ]:
# Final model + predictions
model = LogisticRegression(max_iter=2000, C=1.0, solver='lbfgs', multi_class='multinomial')
model.fit(X_train, y_train)
test_preds = model.predict_proba(X_test)

print(f'Test predictions shape: {test_preds.shape}')

In [ ]:
# Save
np.save(OUTPUT_DIR / 'sbert_oof.npy', oof_preds)
np.save(OUTPUT_DIR / 'sbert_test.npy', test_preds)

del sbert
gc.collect()
torch.cuda.empty_cache() if DEVICE == 'cuda' else None

print('Saved SBERT predictions')